# Huấn luyện NS-MGAT trên Google Colab

> Dùng cho **mọi** thí nghiệm cần GPU: `phobert` (CD1.5), `asgcn` + `asgcn_linked` (CD1.6a),
> `senticgcn` (CD1.6b). Chỉ đổi hai chữ ở ô "Chạy một seed".

**Vì sao không chạy ở máy:** đo thật trên CPU 8 nhân của máy học viên — 31,8 giây/bước,
tức **52,8 giờ cho một seed**, 6,6 ngày cho 3 seed. Colab T4 mất ~30–45 phút/seed.

## Chuẩn bị một lần (làm ở máy, trước khi mở notebook này)

1. **Đẩy code lên GitHub** — Colab kéo code từ đó:
   ```powershell
   git push
   ```
2. **Tải dữ liệu lên Drive.** `data/processed/*.jsonl` không nằm trong git (đã `.gitignore`),
   nên phải đưa lên Drive tay, **một lần duy nhất**:
   - Tạo trên Google Drive thư mục `nsmgat/data/processed/`
   - Tải lên 3 file trong `data/processed/` ở máy: `visfd_train.jsonl`,
     `visfd_dev.jsonl`, `visfd_test.jsonl` (tổng 45 MB)

## Cách notebook này giữ kết quả khi Colab ngắt phiên

Colab xoá sạch máy ảo khi hết phiên. Nên `results/`, `checkpoints/`, `logs/` được **nối
sang Drive bằng symlink** — ghi vào chúng là ghi thẳng vào Drive.

Nối thư mục chứ **không** sửa đường dẫn trong config, vì `results_dir` và `ckpt_dir` nằm
trong `config_hash`: sửa là lần chạy Colab mang vân tay khác với `configs/phobert.yaml`,
và không còn đối chiếu được với các lần chạy khác.

## 1. Kiểm GPU — làm trước mọi thứ khác

In [1]:
# Chay CPU thi mot seed mat ~53 gio. Dung lai ngay con hon chay 3 tieng roi moi biet.
import torch, subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "(khong thay nvidia-smi)")
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "KHONG CO GPU. Vao Runtime > Change runtime type > Hardware accelerator = GPU (T4), "
    "roi chay lai o nay."
)
print("OK — co GPU, chay tiep duoc.")

Tesla T4, 15360 MiB
torch: 2.11.0+cu130 | CUDA: True
OK — co GPU, chay tiep duoc.


## 2. Gắn Drive và dựng cây thư mục

In [2]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

GOC_DRIVE = Path("/content/drive/MyDrive/nsmgat")
for ten in ("data/processed", "results", "checkpoints", "logs"):
    (GOC_DRIVE / ten).mkdir(parents=True, exist_ok=True)

co = sorted(p.name for p in (GOC_DRIVE / "data/processed").glob("*.jsonl"))
print("Thu muc Drive:", GOC_DRIVE)
print("Du lieu da co :", co or "(CHUA CO — xem phan 'Chuan bi mot lan' o dau notebook)")

Mounted at /content/drive
Thu muc Drive: /content/drive/MyDrive/nsmgat
Du lieu da co : ['visfd_dev.jsonl', 'visfd_test.jsonl', 'visfd_train.jsonl']


## 3. Lấy code

Cách chính: kéo từ GitHub. Nếu kho riêng tư hoặc bạn chưa `git push`, dùng ô **3b**.

In [12]:
import subprocess
from pathlib import Path

REPO = Path("/content/nsmgat")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/2611326-NguyenMinhTrong/nsmgat-vietnamese-absa.git", str(REPO)], check=True)

# In ra commit dang chay — de sau nay truy duoc ket qua nay sinh tu ma nguon nao
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

0e41767 GAP-021: dua trang thai RNG ve ByteTensor tren CPU truoc khi khoi phuc



### 3b. *(chỉ khi ô trên không dùng được)* Lấy code từ Drive

Ở máy: nén thư mục repo (bỏ `.venv`, `data`, `results`, `checkpoints`) thành `nsmgat.zip`,
tải lên `Drive/nsmgat/nsmgat.zip`, rồi chạy ô này.

In [ ]:
# !unzip -q -o /content/drive/MyDrive/nsmgat/nsmgat.zip -d /content/nsmgat

## 4. Cài thư viện

Chỉ cài thứ Colab **chưa có**. Không chạy `pip install -r requirements.txt`: file đó có
`torch`, cài đè lên bản torch-CUDA sẵn của Colab là hỏng GPU.

Dòng `pip install -e` là **bắt buộc**, không phải tuỳ chọn: mã nguồn nằm ở `src/nsmgat/`
(src-layout), không cài thì `python -m nsmgat.train` báo `ModuleNotFoundError: nsmgat`.

In [16]:
%pip install -q transformers pyyaml
%pip install -q -e /content/nsmgat   # BAT BUOC — src-layout, khong cai thi khong import duoc

import importlib, nsmgat
importlib.reload(nsmgat)
print("nsmgat import duoc tu:", nsmgat.__file__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nsmgat (pyproject.toml) ... done
nsmgat import duoc tu: None


## 5. Nối thư mục sang Drive

Sau ô này, mọi thứ mô hình ghi ra (`results/`, `checkpoints/`, `logs/`) đi thẳng vào Drive —
Colab ngắt phiên cũng không mất.

In [9]:
import os
from pathlib import Path

REPO = Path("/content/nsmgat")
GOC_DRIVE = Path("/content/drive/MyDrive/nsmgat")
os.chdir(REPO)

for ten in ("data/processed", "results", "checkpoints", "logs"):
    dich = REPO / ten
    dich.parent.mkdir(parents=True, exist_ok=True)
    if dich.is_symlink():
        dich.unlink()
    elif dich.exists():
        # Thu muc that (vd data/processed rong tu git) -> doi ten de khoi mat
        dich.rename(dich.with_name(dich.name + "_local"))
    os.symlink(GOC_DRIVE / ten, dich)
    print(f"{ten:18} -> {GOC_DRIVE / ten}")

# Kiem du lieu that su doc duoc, khong chi la co file
for ten in ("train", "dev", "test"):
    p = REPO / "data/processed" / f"visfd_{ten}.jsonl"
    assert p.exists(), f"THIEU {p} — tai len Drive truoc (xem dau notebook)"
    print(f"  visfd_{ten}: {sum(1 for _ in p.open(encoding='utf-8')):,} dong")

data/processed     -> /content/drive/MyDrive/nsmgat/data/processed
results            -> /content/drive/MyDrive/nsmgat/results
checkpoints        -> /content/drive/MyDrive/nsmgat/checkpoints
logs               -> /content/drive/MyDrive/nsmgat/logs
  visfd_train: 23,872 dong
  visfd_dev: 3,316 dong
  visfd_test: 6,722 dong


## 6. Chạy một seed

Đổi `CONFIG` / `MODEL` khi dùng cho thí nghiệm khác. Mỗi seed ~30–45 phút trên T4.

**Đừng đóng tab.** Colab ngắt phiên sau ~90 phút không tương tác; một seed vừa đủ một
phiên. Bị ngắt giữa chừng thì chạy ô **7** (chạy tiếp), không mất gì.

**Chạy `asgcn_linked` thì phải đặt `EXP_NAME`**, nếu không kết quả ghi đè lên
`results/asgcn/` và mất luôn bản gốc vừa chạy.


In [ ]:
CONFIG = "configs/asgcn.yaml"
MODEL = "asgcn"
SEED = 42

# EXP_NAME: ten thu muc ket qua. De "" thi lay ten MODEL.
# BAT BUOC dat khi chay asgcn_linked, neu khong se GHI DE len results/asgcn/
#   configs/asgcn.yaml        -> EXP_NAME = ""
#   configs/asgcn_linked.yaml -> EXP_NAME = "asgcn_linked"
EXP_NAME = ""

them = f" --exp-name {EXP_NAME}" if EXP_NAME else ""
!python -m nsmgat.train --config {CONFIG} --model {MODEL} --seed {SEED}{them}


2026-10-01 20:38:03 | __main__ | INFO | === BAT DAU asgcn/seed42 | config=configs/asgcn.yaml | device=cuda ===
2026-10-01 20:38:03 | __main__ | INFO | Log ghi tai logs/asgcn/seed42.log — theo doi: python scripts/watch_train.py --exp asgcn --seed 42
Loading weights: 100% 197/197 [00:00<00:00, 23356.36it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

In [4]:
!ls -lh /content/drive/MyDrive/nsmgat/results/asgcn/seed42/ 2>/dev/null || echo "CHUA CO results"
!echo "--- 6 dong cuoi log ---"
!tail -6 /content/drive/MyDrive/nsmgat/logs/asgcn/seed42.log
!echo "--- checkpoint ---"
!ls -lh /content/drive/MyDrive/nsmgat/checkpoints/asgcn/seed42/

CHUA CO results
--- 6 dong cuoi log ---
2026-10-01 20:55:41 | __main__ | INFO | [asgcn/seed42] epoch 2/8 | train_loss=0.3117 | dev_acc=0.8993 | dev_macro_f1=0.835935
2026-10-01 21:05:07 | __main__ | INFO | [asgcn/seed42] epoch 3/8 | train_loss=0.2272 | dev_acc=0.8969 | dev_macro_f1=0.836965
2026-10-01 21:14:14 | __main__ | INFO | [asgcn/seed42] epoch 4/8 | train_loss=0.1743 | dev_acc=0.9065 | dev_macro_f1=0.846652
2026-10-01 21:23:14 | __main__ | INFO | [asgcn/seed42] epoch 5/8 | train_loss=0.1360 | dev_acc=0.9119 | dev_macro_f1=0.855294
2026-10-01 21:32:39 | __main__ | INFO | [asgcn/seed42] epoch 6/8 | train_loss=0.1035 | dev_acc=0.9068 | dev_macro_f1=0.845545
2026-10-01 21:41:38 | __main__ | INFO | [asgcn/seed42] epoch 7/8 | train_loss=0.0775 | dev_acc=0.9062 | dev_macro_f1=0.844609
--- checkpoint ---
total 2.1G
-rw------- 1 root root 520M Oct  1 21:24 best.pt
-rw------- 1 root root 1.6G Oct  1 21:45 last.pt


In [20]:
from pathlib import Path

GOC = Path("/content/drive/MyDrive/nsmgat")
tong = 0
for exp in sorted(p for p in (GOC / "checkpoints").iterdir() if p.is_dir()):
    for seed in sorted(p for p in exp.iterdir() if p.is_dir()):
        xong = (GOC / "results" / exp.name / seed.name / "metrics.json").exists()
        for f in sorted(seed.glob("*.pt")):
            gb = f.stat().st_size / 1e9
            tong += gb
            print(f"{gb:6.2f} GB  {'XONG     ' if xong else 'CHUA XONG'}  {exp.name}/{seed.name}/{f.name}")
print(f"\nTong checkpoints: {tong:.2f} GB")

  0.54 GB  XONG       asgcn/seed2024/best.pt
  1.63 GB  XONG       asgcn/seed2024/last.pt

Tong checkpoints: 2.17 GB


In [18]:
from pathlib import Path

GOC = Path("/content/drive/MyDrive/nsmgat")
can_xoa = []
for f in sorted((GOC / "checkpoints").rglob("last.pt")):
    exp, seed = f.parent.parent.name, f.parent.name
    if (GOC / "results" / exp / seed / "metrics.json").exists():
        can_xoa.append(f)
        print(f"{f.stat().st_size / 1e9:5.2f} GB  {exp}/{seed}/last.pt")
print(f"\n{len(can_xoa)} file, {sum(f.stat().st_size for f in can_xoa) / 1e9:.2f} GB")

# Doc ky danh sach tren. Dung y roi thi bo dau # o dong duoi va chay lai:
# for f in can_xoa: f.unlink()

 1.63 GB  asgcn/seed1337/last.pt

1 file, 1.63 GB


## 7. Chạy tiếp sau khi bị ngắt

`--resume` đọc `checkpoints/<exp>/seed<N>/last.pt` trên Drive và chạy tiếp **đúng chỗ đang
dở**, gồm cả trạng thái sinh số ngẫu nhiên — kết quả giống hệt chạy liền mạch
(`tests/test_resume.py` khoá điều này).

Đổi config rồi thì nó **từ chối** chạy tiếp: ghép nửa đầu cấu hình này với nửa sau cấu
hình khác thì con số thu được vô nghĩa.

In [13]:
!python -m nsmgat.train --config configs/asgcn.yaml --model asgcn --seed 42 --resume


2026-10-02 01:19:40 | __main__ | INFO | === BAT DAU asgcn/seed42 | config=configs/asgcn.yaml | device=cuda ===
2026-10-02 01:19:40 | __main__ | INFO | Log ghi tai logs/asgcn/seed42.log — theo doi: python scripts/watch_train.py --exp asgcn --seed 42
Loading weights: 100% 197/197 [00:00<00:00, 22342.70it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

## 8. Hai seed còn lại

In [17]:
!python -m nsmgat.train --config configs/asgcn.yaml --model asgcn --seed 1337

2026-10-02 01:47:06 | __main__ | INFO | === BAT DAU asgcn/seed1337 | config=configs/asgcn.yaml | device=cuda ===
2026-10-02 01:47:06 | __main__ | INFO | Log ghi tai logs/asgcn/seed1337.log — theo doi: python scripts/watch_train.py --exp asgcn --seed 1337
Loading weights: 100% 197/197 [00:00<00:00, 26148.86it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream

In [19]:
!python -m nsmgat.train --config configs/asgcn.yaml --model asgcn --seed 2024

2026-10-02 03:00:45 | __main__ | INFO | === BAT DAU asgcn/seed2024 | config=configs/asgcn.yaml | device=cuda ===
2026-10-02 03:00:45 | __main__ | INFO | Log ghi tai logs/asgcn/seed2024.log — theo doi: python scripts/watch_train.py --exp asgcn --seed 2024
Loading weights: 100% 197/197 [00:00<00:00, 17602.48it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream

In [ ]:
for seed in (1337, 2024):
    print()
    print("=" * 70)
    print(f"  SEED {seed}")
    print("=" * 70)
    !python -m nsmgat.train --config {CONFIG} --model {MODEL} --seed {seed}{them}


## 9. Xem kết quả 3 seed

In [ ]:
import json, statistics
from pathlib import Path

EXP = EXP_NAME or MODEL   # khop voi --exp-name o o 6
hang = []
for seed in (42, 1337, 2024):
    p = Path("results") / EXP / f"seed{seed}" / "metrics.json"
    if not p.exists():
        print(f"seed {seed}: chua co ket qua")
        continue
    m = json.loads(p.read_text(encoding="utf-8"))
    hang.append((seed, m["test"]["accuracy"], m["test"]["macro_f1"],
                 m["test"]["f1_per_class"], m.get("config_hash", "?")[:12],
                 m.get("train_time_sec", 0)))

print(f"{'seed':>6} {'acc':>8} {'macro-F1':>9} {'F1(NEU)':>8} {'van tay':>14} {'phut':>7}")
for seed, acc, f1, per_class, h, t in hang:
    print(f"{seed:>6} {acc:>8.4f} {f1:>9.4f} {per_class[1]:>8.3f} {h:>14} {t/60:>7.1f}")

if len(hang) == 3:
    for ten, i in (("accuracy", 1), ("macro-F1", 2)):
        gia_tri = [h[i] for h in hang]
        print(f"\n{ten}: {statistics.mean(gia_tri):.4f} +/- {statistics.stdev(gia_tri):.4f}")
    print("")
    print("So sanh (accuracy, trung binh 3 seed):")
    print("  lexicon 0,7469 | bilstm 0,8642 | phobert 0,9186")
    print("  Hieu so voi phobert la con so tra loi: cu phap dang may diem.")


## 10. Mang kết quả về máy

`results/` đã nằm trên Drive rồi. Ở **máy**, chép vào repo cho đúng chỗ:

```
Drive/nsmgat/results/phobert/seed42/{metrics.json, predictions.jsonl}
   →  <repo>/results/phobert/seed42/
```

Chỉ cần **`metrics.json`** và **`predictions.jsonl`**. **Đừng chép `checkpoints/`** vào repo:
`best.pt` của phobert nặng ~540 MB, và `.gitignore` đã chặn `checkpoints/`.

Về máy rồi thì kiểm lại bằng:

```powershell
.\.venv\Scripts\python.exe -m pytest -q
```

## 11. Dọn Drive khi chạy xong cả 3 seed

`last.pt` của phobert nặng **~1,6 GB mỗi seed** (trọng số + 2 trạng thái Adam). Ba seed là
gần 5 GB — Drive miễn phí có 15 GB.

Xong hẳn rồi thì xoá `last.pt`; **giữ `best.pt`** nếu còn muốn chạy lại đánh giá
(`--no-train`). Ô dưới chỉ *liệt kê* — muốn xoá thì bỏ dấu `#` ở dòng cuối.

In [ ]:
from pathlib import Path

for p in sorted(Path("/content/drive/MyDrive/nsmgat/checkpoints").rglob("*.pt")):
    print(f"{p.stat().st_size/1e9:6.2f} GB  {p}")

# for p in Path("/content/drive/MyDrive/nsmgat/checkpoints").rglob("last.pt"): p.unlink()